# Exploratory Data Analysis (EDA) per Outlet
Analisis awal data ulasan, rating, dan sentimen untuk setiap cabang Kopi Kenangan.

In [ ]:
# Setup environment jika dijalankan di Google Colab / Kaggle
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

if IN_COLAB:
    print("Colab detected. Pastikan file 'cleaned_reviews.csv' sudah diupload.")
elif IN_KAGGLE:
    print("Kaggle detected. Sesuaikan input path dataset.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
# Cari path lokal atau fallback ke Colab/Kaggle
dataset_path = '../../preprocessing/cleaned_reviews.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'cleaned_reviews.csv'

df = pd.read_csv(dataset_path)
df = df.dropna(subset=['text_clean'])
print(f"Total data: {len(df)} baris")
df.head()

## 2. Preprocessing Kolom Rating

In [ ]:
# Konversi string rating ke numerik
df['rating_num'] = df['rating'].str.extract('(\d+)').astype(float)
print(f"Jumlah outlet: {df['nama_gerai'].nunique()}")

## 3. Agregasi Data per Outlet

In [ ]:
# Grouping statistik per gerai
summary_per_gerai = df.groupby('nama_gerai').agg(
    total_review=('ulasan', 'count'),
    avg_rating=('rating_num', 'mean'),
    avg_word_count=('word_count', 'mean')
).reset_index()

# Hitung distribusi sentimen
sentiment_dist = pd.crosstab(df['nama_gerai'], df['sentiment'], normalize='index') * 100
sentiment_dist = sentiment_dist.rename(columns={
    'negative': 'pct_negative',
    'neutral': 'pct_neutral',
    'positive': 'pct_positive'
}).reset_index()

gerai_stats = pd.merge(summary_per_gerai, sentiment_dist, on='nama_gerai')

# Filter/flag outlet dengan review < 20 (low confidence)
gerai_stats['confidence_level'] = gerai_stats['total_review'].apply(lambda x: 'High' if x >= 20 else 'Low')
gerai_stats = gerai_stats.sort_values(by='total_review', ascending=False)
gerai_stats

## 4. Visualisasi

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=gerai_stats, y='nama_gerai', x='total_review', hue='confidence_level', palette={'High': 'teal', 'Low': 'salmon'})
plt.title('Jumlah Ulasan Per Gerai Kopi Kenangan')
plt.xlabel('Jumlah Ulasan')
plt.ylabel('Nama Gerai')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=gerai_stats.sort_values(by='avg_rating', ascending=False), y='nama_gerai', x='avg_rating', color='orange')
plt.axvline(x=4.0, color='red', linestyle='--', label='Threshold 4.0')
plt.title('Rata-rata Rating Per Gerai Kopi Kenangan')
plt.xlabel('Rata-rata Rating')
plt.ylabel('Nama Gerai')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Simpan Hasil Analisis

In [ ]:
os.makedirs('../reports', exist_ok=True)

report_dict = {
    "total_reviews_all_outlets": int(len(df)),
    "total_outlets": int(df['nama_gerai'].nunique()),
    "outlets_stats": gerai_stats.to_dict(orient='records'),
    "low_confidence_outlets": gerai_stats[gerai_stats['confidence_level'] == 'Low']['nama_gerai'].tolist()
}

with open('../reports/eda_per_gerai_summary.json', 'w') as f:
    json.dump(report_dict, f, indent=4)

print("Laporan berhasil disimpan di modeling/reports/")